In [17]:
import pandas as pd

df_val = pd.read_csv("raw_data/player_valuations.csv")
df_val.head()

,player_id,date,market_value_in_eur,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,3057,BE1
1,342216,2001-07-20,100000,1241,SC1
2,3132,2003-12-09,400000,126,TR1
3,6893,2003-12-15,900000,984,GB1
4,10,2004-10-04,7000000,398,IT1


In [18]:
df_val = df_val.sort_values(['player_id', 'date'])

In [19]:
df_val.columns = (
    df_val.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w\s]", "", regex=True)
)

In [20]:
df_val['date'] = pd.to_datetime(df_val['date'], errors='coerce')

In [21]:
df_val = df_val.dropna(subset=['date'])

In [22]:
# checking for duplicates
duplicate_count = df_val.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

# dropping duplicates
df_val = df_val.drop_duplicates()

Duplicate rows: 0


In [23]:
# summary of missing values
missing_summary = df_val.isna().sum()
print(missing_summary)

player_id                              0
date                                   0
market_value_in_eur                    0
current_club_id                        0
player_club_domestic_competition_id    0
dtype: int64


In [24]:
# checking for negatives or unrealistic values
print(df_val['market_value_in_eur'].describe())
print(df_val['player_id'].nunique(), 'unique players')
print(df_val['date'].min(), 'to', df_val['date'].max())

count    4.966060e+05
mean     2.471145e+06
std      6.983759e+06
min      0.000000e+00
25%      2.000000e+05
50%      5.000000e+05
75%      1.700000e+06
max      2.000000e+08
Name: market_value_in_eur, dtype: float64
31078 unique players
2000-01-20 00:00:00 to 2025-04-06 00:00:00


In [25]:
# checking for unrealistic low values
df_val[df_val['market_value_in_eur'] < 1000]

,player_id,date,market_value_in_eur,current_club_id,player_club_domestic_competition_id
15331,60096,2008-03-17,0,2696,RU1


In [26]:
#earliest and latest date in the dataset
print(df_val['date'].min())
print(df_val['date'].max())

df_val['date'] = pd.to_datetime(df_val['date'])

#season label function
def get_season_label(date):
    year = date.year
    if date.month >= 7:
        return f"{str(year)[2:]}/{str(year + 1)[2:]}"
    else:
        return f"{str(year - 1)[2:]}/{str(year)[2:]}"

#insert season label column
df_val['season'] = df_val['date'].apply(get_season_label)

2000-01-20 00:00:00
2025-04-06 00:00:00


In [27]:
#keeping only relevant columns
df_val = df_val[['player_id', 'date', 'market_value_in_eur', 'season']]

df_val.head(20)

,player_id,date,market_value_in_eur,season
4,10,2004-10-04,7000000,04/05
2326,10,2005-01-07,9000000,04/05
3124,10,2005-05-05,12000000,04/05
4017,10,2005-09-30,15000000,05/06
4755,10,2006-01-09,20000000,05/06
6254,10,2006-07-15,30000000,06/07
9024,10,2007-06-21,23000000,06/07
16465,10,2008-06-04,20000000,07/08
26702,10,2009-06-10,18000000,08/09
29922,10,2009-08-30,12000000,09/10


In [28]:
import os

# create the directory
os.makedirs("to_merge_data", exist_ok=True)

In [29]:
df_val.to_csv("to_merge_data/j2_val.csv", index=False)